In [1]:
# 1. Traer TU rama del repo e instalar el paquete
!git clone -b silva/dia1-io https://github.com/gabriela708/ML-Household-Socioeconomic-Status.git repo
%cd repo
!pip install -q -e . pyyaml pyarrow
import sys; sys.path.insert(0, "src")

Cloning into 'repo'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 152 (delta 41), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 8.44 MiB | 6.29 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ses (pyproject.toml) ... done


In [2]:
# 2. Leer los 8 módulos y verificar llaves
from ses.io import read_module, make_hogar_id, check_keys
from ses.config import CFG

resumen = []
for nombre, spec in CFG["raw_files"].items():
    df = read_module(nombre)
    check_keys(df, spec["nivel"])
    cab = df[df["CLASE"] == 1]
    resumen.append({"modulo": nombre, "nivel": spec["nivel"], "filas": len(df),
                    "hogares": make_hogar_id(df).nunique(),
                    "hogares_cabecera": make_hogar_id(cab).nunique()})
import pandas as pd
pd.DataFrame(resumen)

,modulo,nivel,filas,hogares,hogares_cabecera
0,hogar_vivienda,hogar,24159,24159,21037
1,personas_salud_educacion,persona,66559,24159,21037
2,fuerza_trabajo,persona,53461,24158,21036
3,ocupados,persona,29611,19594,17040
4,no_ocupados,persona,23850,15670,13586
5,otras_formas_trabajo,persona,53461,24158,21036
6,otros_ingresos_impuestos,persona,53461,24158,21036
7,migracion,persona,66559,24159,21037


In [3]:
# 3. Población del proyecto: hogares de cabecera
h = read_module("hogar_vivienda")
h["hogar_id"] = make_hogar_id(h)
hc = h[h["CLASE"] == 1]
print("Hogares totales:", len(h), "| cabecera:", len(hc))
hc["DPTO"].value_counts().head(10)

Hogares totales: 24159 | cabecera: 21037


,count
DPTO,
05,1302
76,1118
66,1088
17,1082
68,993
52,942
63,932
50,848
11,807


Ocupados cubre 17.040 hogares de cabecera y no ocupados 13.586, así que al unir esos módulos al hogar habrá hogares sin registros

DANE – Catálogo 908, Pobreza Monetaria y Desigualdad 2025, Se sube este archivo para poder hacer cruces y analisis iniciales de que cumpla para nuestro objetivo y que funcione bien con nuestro archivo .io.

In [5]:
import pandas as pd
Hogares=pd.read_csv("/content/Hogares.csv")

In [6]:
Hogares.head()

,directorio,secuencia_p,mes,nper,npersug,clase,ingtotug,dominio,lp,li,...,ingtotarr,ingpcug,pobre,indigente,npobres,nindigentes,fex_c,ciudad_sinam,dpto,fex_dpto
0,8086872,1,1,1,1,2,1.040000e+06,RURAL,308654.56309,182935.52974,...,1.390000e+06,1.390000e+06,0,0,0,0,182.062690,NaN,Bolívar,80.336320
1,8086874,1,1,3,3,2,2.320833e+06,RURAL,308654.56309,182935.52974,...,2.520833e+06,8.402778e+05,0,0,0,0,100.707518,NaN,Bolívar,64.573496
2,8086875,1,1,1,1,2,2.784000e+06,RURAL,308654.56309,182935.52974,...,3.084000e+06,3.084000e+06,0,0,0,0,175.342825,NaN,Bolívar,68.999560
3,8086876,1,1,3,3,2,2.686667e+06,RURAL,308654.56309,182935.52974,...,2.986667e+06,9.955556e+05,0,0,0,0,201.672077,NaN,Bolívar,145.284264
4,8086877,1,1,3,3,2,3.385333e+06,RURAL,308654.56309,182935.52974,...,3.685333e+06,1.228444e+06,0,0,0,0,208.939282,NaN,Bolívar,122.440831


Se observa que las llaves pricipales son "directorio" y "secuencia_p", más no trae una columna "HOGAR" como en nuestors otros datos, asi que a primera vista no se podria hacer el cruce (en diciembre) pero el comportamiento de las dos llaves es igual al de "HOGAR" por lo tanto el cruce podria realiazarse con estas dos y se observa (segun  el diccionario del DANE) tiene los 12 meses del año asi que, si logramos cumplir nuestro plan principal de hacerlo con esta base enfocada en el mes de diciembre.

Clase hace referencia si es cabezera o otro, donde 1 es cabecera y 2 cualquier otro

In [9]:
# B. Columnas, tipos y faltantes
Hogares.info()
Hogares[["mes", "clase"]].value_counts().sort_index()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 276666 entries, 0 to 276665
Data columns (total 23 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   directorio    276666 non-null  int64  
 1   secuencia_p   276666 non-null  int64  
 2   mes           276666 non-null  int64  
 3   nper          276666 non-null  int64  
 4   npersug       276666 non-null  int64  
 5   clase         276666 non-null  int64  
 6   ingtotug      276666 non-null  float64
 7   dominio       276666 non-null  object 
 8   lp            276666 non-null  float64
 9   li            276666 non-null  float64
 10  p5090         276666 non-null  int64  
 11  p5100         9241 non-null    float64
 12  p5130         164614 non-null  float64
 13  ingtotarr     276666 non-null  float64
 14  ingpcug       276666 non-null  float64
 15  pobre         276666 non-null  int64  
 16  indigente     276666 non-null  int64  
 17  npobres       276666 non-null  int64  
 18  nind

mes  clase
1    1        19702
     2         3089
2    1        19878
     2         3102
3    1        19770
     2         3073
4    1        19791
     2         3084
5    1        20030
     2         3056
6    1        19876
     2         3111
7    1        20012
     2         3126
8    1        20144
     2         3118
9    1        20068
     2         3088
10   1        20217
     2         3172
11   1        20232
     2         3115
12   1        19690
     2         3122
Name: count, dtype: int64

In [11]:
# C. Diciembre + cabecera, y unicidad de la llave
Hogares_12 = Hogares[(Hogares["mes"].astype(int) == 12) & (Hogares["clase"].astype(int) == 1)].copy()
print("Hogares cabecera diciembre (DANE):", len(Hogares_12))
print("Llaves duplicadas:", Hogares_12.duplicated(["directorio", "secuencia_p"]).sum())

Hogares cabecera diciembre (DANE): 19690
Llaves duplicadas: 0


FUCKKK NO ME SALE EL PUTO CRUCE

In [12]:
# D. El cruce con nuestros hogares de cabecera (hc viene de tu celda 3)
Hogares_12["hogar_id"] = (Hogares_12["directorio"].astype(int).astype(str) + "_" +
                    Hogares_12["secuencia_p"].astype(int).astype(str) + "_" +
                    Hogares_12["secuencia_p"].astype(int).astype(str))   # HOGAR == SECUENCIA_P

cruce = hc.merge(Hogares_12, on="hogar_id", how="left", indicator=True)
print(cruce["_merge"].value_counts())
print("Cruzan:", (cruce["_merge"] == "both").sum(), "de", len(hc))

_merge
both          19690
left_only      1347
right_only        0
Name: count, dtype: int64
Cruzan: 19690 de 21037


Se cruzan almenos un 93% de los hogares de nuestra base de datos del DANE, por lo que podemos proceder sin problemas, pero tenemos que concluir que pasa con el otro 7%

In [13]:
# E. Calidad del ingreso que usaremos para el target
m = cruce[cruce["_merge"] == "both"]
print(m["ingpcug"].describe())
print("ingpcug faltante:", m["ingpcug"].isna().sum(), "| ingpcug = 0:", (m["ingpcug"] == 0).sum())
print("Valores de lp:", m["lp"].nunique(), m["lp"].unique()[:10])
print(m["pobre"].value_counts(normalize=True))

count    1.969000e+04
mean     1.605759e+06
std      2.023790e+06
min      0.000000e+00
25%      5.752292e+05
50%      1.039202e+06
75%      1.887500e+06
max      7.250000e+07
Name: ingpcug, dtype: float64
ingpcug faltante: 0 | ingpcug = 0: 72
Valores de lp: 24 [437134.13469 604479.52799 502467.41445 664042.8997  526866.19846
 621055.90475 549545.57777 543417.5585  620072.94077 492316.85917]
pobre
0.0    0.78385
1.0    0.21615
Name: proportion, dtype: float64


### Conclusiones sobre la Calidad del Ingreso (Sección E)

Tras analizar los datos cruzados con el archivo del DANE (`_merge == 'both'`), se destacan los siguientes puntos clave sobre nuestra variable objetivo (**target**):

1. Distribución del Ingreso (`ingpcug`):
   * El ingreso per cápita promedio es de 1.605.759 COP, con una mediana sustancialmente menor (1.039.202 COP). Esto refleja una asimetría positiva esperada (sesgo hacia la derecha), típica en la distribución de ingresos.
   * El valor máximo registrado es de 72.500.000 COP, mientras que el mínimo es 0 COP.

2. **Calidad de los Datos:**
   * **Sin datos faltantes:** No se presentan valores nulos (`NaN`) en la columna `ingpcug` para los hogares cruzados.
   * **Ingresos en cero:** Solo **72 hogares** registran un ingreso de exactamente cero pesos ($0), lo cual representa un porcentaje mínimo e ideal que no afecta la robustez de los datos.

3. **Líneas de Pobreza y Clasificación:**
   * Se identifican **24 valores distintos para la línea de pobreza (`lp`)**, reflejando que este umbral está correctamente adaptado a la composición y ubicación de cada hogar.
   * El **21.61%** de los hogares se encuentran bajo la línea de pobreza (`pobre = 1`), mientras que el **78.39%** están clasificados como no pobres. Esta distribución de clases está bien balanceada para entrenar modelos predictivos de clasificación.

In [14]:
# F1. ¿Cuántos hogares de diciembre-cabecera trae el DANE, y cuántos de ellos no están en hc?
print("DANE dic-cabecera:", len(Hogares_12))
print("DANE que no están en hc:", (~Hogares_12["hogar_id"].isin(hc["hogar_id"])).sum())

DANE dic-cabecera: 19690
DANE que no están en hc: 0


Se puede decir que el problema no es que no esten en HC si no puede ser otraz razones

In [17]:
# F2. Los 1.347 que no cruzan: ¿están en el archivo DANE con otro mes u otra clase?
Hogares["hogar_id"] = (Hogares["directorio"].astype(int).astype(str) + "_" +
                  Hogares["secuencia_p"].astype(int).astype(str) + "_" +
                  Hogares["secuencia_p"].astype(int).astype(str))
perdidos = cruce.loc[cruce["_merge"] == "left_only", "hogar_id"]
enDANE = Hogares[Hogares["hogar_id"].isin(perdidos)]
print("Perdidos que sí están en el archivo DANE completo:", enDANE["hogar_id"].nunique())
print(enDANE[["mes", "clase"]].value_counts())

Perdidos que sí están en el archivo DANE completo: 0
Series([], Name: count, dtype: int64)


Tambien se observa, que no es que esten en otro mes distinto a diciembre o que no sean cabeceras de hogar

In [18]:
# F3. ¿Dónde están los perdidos? (si se concentran en pocos departamentos, hay sesgo)
cruce["cruza"] = cruce["_merge"] == "both"
t = cruce.groupby("DPTO")["cruza"].agg(["size", "mean"]).sort_values("mean")
t.head(10)

,size,mean
DPTO,,
81,133,0.0
85,165,0.0
86,143,0.0
88,266,0.0
91,144,0.0
94,106,0.0
95,147,0.0
97,155,0.0
99,88,0.0


Ya luego de revisar la razon de esas observaciones que no se cruzan, vemos que no estan como tal de esta base del DANE asi que podemos decir que los excluyo con sentido, luego se ve que al revisar donde hacen falta vemos que son especificamente en 9 departamentos: Esos nueve son San Andrés (88) y los ocho departamentos de Orinoquía y Amazonía: Arauca (81), Casanare (85), Putumayo (86), Amazonas (91), Guainía (94), Guaviare (95), Vaupés (97) y Vichada (99). El DANE los encuesta en la GEIH, pero en su archivo de pobreza monetaria de 2025 no aparecen. No es una pérdida al azar: el DANE los dejó por fuera de esa base. (REVISAR EN EL ARTICULO SI HAY RAZON)

### SUBIR EL CSV SOLO FILTRADO POR DICIEMBRE:

In [20]:
Hogares_12_full = Hogares[Hogares["mes"].astype(int) == 12]   # diciembre, cabecera y resto (el filtro de clase se hace en código)
cols = ["directorio", "secuencia_p", "mes", "clase", "dominio", "dpto", "nper", "npersug", "ingpcug", "li", "lp", "pobre", "indigente", "fex_c"]
Hogares_12_full[cols].to_csv("pobreza_monetaria_2025_hogares.csv", index=False)

In [21]:
import os
from google.colab import files

ruta = "pobreza_monetaria_2025_hogares.csv"
print("Carpeta actual:", os.getcwd())
print("Tamaño:", round(os.path.getsize(ruta) / 1e6, 2), "MB")
print(pd.read_csv(ruta).shape)   # deberían ser solo filas de diciembre
files.download(ruta)

Carpeta actual: /content/repo
Tamaño: 2.08 MB
(22812, 14)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>